In [1]:
#@title Setup -- ablacoes (D vs D-1, janelas de lookback)
from pathlib import Path
PROJECT = Path(r'D:\tcc_ibov')
DIRS = {k: PROJECT / v for k, v in {
    'raw': 'data/raw', 'processed': 'data/processed',
    'models': 'models', 'reports': 'reports', 'utils': 'utils',
}.items()}

import sys; sys.path.insert(0, str(DIRS['utils']))
import importlib, tcc_utils; importlib.reload(tcc_utils)
from tcc_utils import make_target, walk_forward_splits

import random, os, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (mean_absolute_error, f1_score,
                             matthews_corrcoef, brier_score_loss)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

class LSTMRegressor(nn.Module):
    """Identico ao nivel 4/5 (notebook 05): 1 camada, hidden=16, dropout=0.4."""
    def __init__(self, n_features, hidden=16, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.lstm(x)
        last = self.dropout(out[:, -1, :])
        return self.head(last).squeeze(-1)

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

def treina_avalia(X_tr, y_tr, X_te, seed):
    set_seed(seed)
    model = LSTMRegressor(X_tr.shape[-1], HIDDEN, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()
    Xtr = torch.tensor(X_tr, device=DEVICE); ytr = torch.tensor(y_tr, device=DEVICE)
    n = len(Xtr)
    model.train()
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)
        for i in range(0, n, BATCH):
            b = perm[i:i+BATCH]
            opt.zero_grad()
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_te, device=DEVICE)).cpu().numpy()
    return pred

def roda_lstm(X_day_full, y_raw_full, index_full, lookback, tag, k_pca=None, X_emb_full=None):
    """Pipeline generico: se k_pca/X_emb_full forem passados, aplica PCA por
    fold sobre X_emb_full e concatena com X_day_full (preco [+ sig]) antes de
    sequenciar. Caso contrario, X_day_full ja e o tensor diario completo.
    embargo = lookback (regra do projeto p/ qualquer janela testada)."""
    splits = walk_forward_splits(index_full, n_splits=5, min_train=252,
                                  expanding=True, embargo=lookback)
    registros, preds_media, var_exp = [], {}, []
    for kf, (tr_days, te_days) in enumerate(splits, 1):
        t0 = time.time()
        tr_set, te_set = set(tr_days), set(te_days)

        if k_pca is not None:
            tr_day_mask = np.array([i in tr_set for i in range(len(X_emb_full))])
            pca = PCA(n_components=k_pca, random_state=0).fit(X_emb_full[tr_day_mask])
            var_exp.append(pca.explained_variance_ratio_.sum())
            X_emb_pca = pca.transform(X_emb_full).astype(np.float32)
            X_day = np.concatenate([X_day_full, X_emb_pca], axis=1)
        else:
            X_day = X_day_full

        seqs, targets, sidx = [], [], []
        for i in range(lookback - 1, len(X_day)):
            seqs.append(X_day[i-lookback+1:i+1]); targets.append(y_raw_full[i]); sidx.append(i)
        seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

        mask_tr = np.array([i in tr_set for i in sidx])
        mask_te = np.array([i in te_set for i in sidx])
        X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
        X_te, y_te = seqs[mask_te], targets[mask_te]
        if len(X_tr) == 0 or len(X_te) == 0:
            print(f"  fold {kf}: sem dados suficientes (lookback={lookback}) -- pulando")
            continue

        n_f = X_tr.shape[-1]
        sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
        X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
        X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

        n_cal = max(int(len(X_tr) * 0.2), 1)
        preds_seeds = []
        for s in SEEDS:
            pred = treina_avalia(X_tr, y_tr, X_te, s)
            preds_seeds.append(pred)
            pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
            iso = IsotonicRegression(out_of_bounds='clip').fit(
                pred_tr, (y_tr[-n_cal:] > 0).astype(int))
            p_prob = iso.predict(pred)
            y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
            registros.append({
                'fold': kf, 'seed': s,
                'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
                'DA': da(y_te, pred),
                'F1': f1_score(y_bin, pred_dir, zero_division=0),
                'MCC': matthews_corrcoef(y_bin, pred_dir),
                'Brier': brier_score_loss(y_bin, p_prob),
            })
        preds_media[kf] = np.mean(preds_seeds, axis=0)
        print(f"  fold {kf}: {len(X_tr)} tr seqs | {time.time()-t0:.0f}s")

    df = pd.DataFrame(registros)
    np.savez(DIRS['reports'] / f'preds_{tag}.npz',
             **{f'lstm_f{k}': v for k, v in preds_media.items()})
    df.to_parquet(DIRS['reports'] / f'{tag}_full.parquet')
    return df

metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']
def resumo_desvio(df):
    r = df.groupby('fold')[metric_cols].mean().mean()
    d = ((df.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5
    return r, d

print("Setup OK.")


Device: cuda
Setup OK.


In [2]:
#@title Ablacao -- texto do dia D vs. D-1 (defasado), LSTM + BERTimbau k=8
# Testa se usar a noticia de ONTEM (D-1) para prever o retorno de D+1, em vez
# da noticia de HOJE (D), muda o resultado. Espera-se: ainda mais fraco que
# D, ja que o achado central (Secao 5.3 do texto) mostra que o texto de D
# correlaciona com o retorno de D (contemporaneo) -- textos mais antigos
# devem estar ainda mais dessincronizados do alvo D+1.
LOOKBACK_D1 = 21
K_PCA_D1 = 8

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_texto = pd.read_parquet(DIRS['processed'] / 'features_texto_bertimbau.parquet')
feat_texto = feat_texto.reindex(feat_preco.index)

EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']

# Defasagem: texto[i] passa a ser o texto do dia i-1 (shift(1)); descarta o
# primeiro dia (sem D-1 valido) -- sem inventar dado, so reduz N em 1.
feat_texto_lag1 = feat_texto.shift(1)
valid = feat_texto_lag1['emb_000'].notna()
feat_preco_lag = feat_preco.loc[valid]
feat_texto_lag1 = feat_texto_lag1.loc[valid]
print(f"Dias apos remover o 1o (sem D-1): {len(feat_preco_lag)} "
      f"(esperado {len(feat_preco)-1})")

X_price_lag = feat_preco_lag[PRICE_COLS].values.astype(np.float32)
X_sig_lag   = feat_texto_lag1[SIG_COLS].values.astype(np.float32)
X_emb_lag   = feat_texto_lag1[EMB_COLS].values.astype(np.float32)
y_lag       = feat_preco_lag['y'].values.astype(np.float32)
X_day_lag   = np.concatenate([X_price_lag, X_sig_lag], axis=1)

print("=== LSTM + BERTimbau k=8, texto DEFASADO (D-1) ===")
reg_lag1 = roda_lstm(X_day_lag, y_lag, feat_preco_lag.index, LOOKBACK_D1,
                      'lstm_bertimbau_k8_lagD1', k_pca=K_PCA_D1, X_emb_full=X_emb_lag)

r_lag1, d_lag1 = resumo_desvio(reg_lag1)

# Correlacao direta: texto(D-1) -> retorno(D+1) [preditivo, defasado 2 dias
# do alvo] -- comparavel a texto(D) -> retorno(D+1) ja reportado no texto.
corr_lag1 = {}
for c in ['n_noticias', 'emb_dispersao']:
    corr_lag1[c] = np.corrcoef(feat_texto_lag1[c].values, y_lag)[0, 1]
print("\nCorrelacao texto(D-1) -> retorno(D+1):")
for c, v in corr_lag1.items():
    print(f"  {c}: {v:+.4f}")

n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
n5 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
res_n4, _ = resumo_desvio(n4)
res_n5, dev_n5 = resumo_desvio(n5)

print("\n=== MCC: preco | texto-D (nivel 5) | texto-D-1 (esta ablacao) ===")
print(f"  so-preco:      {res_n4['MCC']:+.4f}")
print(f"  texto D (k8):  {res_n5['MCC']:+.4f} +/- {dev_n5['MCC']:.4f}")
print(f"  texto D-1(k8): {r_lag1['MCC']:+.4f} +/- {d_lag1['MCC']:.4f}")
print("\nSalvos: preds_lstm_bertimbau_k8_lagD1.npz, lstm_bertimbau_k8_lagD1_full.parquet")


Dias apos remover o 1o (sem D-1): 1491 (esperado 1491)
=== LSTM + BERTimbau k=8, texto DEFASADO (D-1) ===


  fold 1: 232 tr seqs | 23s


  fold 2: 438 tr seqs | 33s


  fold 3: 644 tr seqs | 50s


  fold 4: 850 tr seqs | 59s


  fold 5: 1056 tr seqs | 70s



Correlacao texto(D-1) -> retorno(D+1):
  n_noticias: -0.0231
  emb_dispersao: -0.0356

=== MCC: preco | texto-D (nivel 5) | texto-D-1 (esta ablacao) ===
  so-preco:      -0.0086
  texto D (k8):  -0.0180 +/- 0.0538
  texto D-1(k8): -0.0158 +/- 0.0537

Salvos: preds_lstm_bertimbau_k8_lagD1.npz, lstm_bertimbau_k8_lagD1_full.parquet


In [3]:
#@title Ablacao -- janelas de lookback {1,3,5,10} -- LSTM so-preco
# O nivel 4 (notebook 05) fixou lookback=21 (alinhado as features de vol
# mais longas e a ACF). Aqui testamos se um lookback mais curto muda a
# conclusao de que o LSTM so-preco nao bate o acaso.
feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']
X_price = feat_preco[PRICE_COLS].values.astype(np.float32)
y_preco = feat_preco['y'].values.astype(np.float32)

sweep_rows = []
for lb in [1, 3, 5, 10]:
    print(f"=== LSTM so-preco, lookback={lb} ===")
    df = roda_lstm(X_price, y_preco, feat_preco.index, lb, f'lstm_preco_lb{lb}')
    r, d = resumo_desvio(df)
    sweep_rows.append({'lookback': lb, 'fonte': 'so-preco', **r.to_dict(),
                        **{f'{m}_sd': d[m] for m in metric_cols}})

# referencia: lookback=21 ja calculado no notebook 05
n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
r21, d21 = resumo_desvio(n4)
sweep_rows.append({'lookback': 21, 'fonte': 'so-preco', **r21.to_dict(),
                    **{f'{m}_sd': d21[m] for m in metric_cols}})

sweep_preco_df = pd.DataFrame(sweep_rows).sort_values('lookback').set_index('lookback')
print("\n=== Sweep de lookback -- LSTM so-preco ===")
print(sweep_preco_df[['MAE','RMSE','DA','MCC','MCC_sd']].round(4).to_string())
sweep_preco_df.to_parquet(DIRS['reports'] / 'sweep_lookback_preco.parquet')
print("\nSalvo: sweep_lookback_preco.parquet")


=== LSTM so-preco, lookback=1 ===


  fold 1: 252 tr seqs | 17s


  fold 2: 458 tr seqs | 30s


  fold 3: 664 tr seqs | 41s


  fold 4: 870 tr seqs | 56s


  fold 5: 1076 tr seqs | 70s
=== LSTM so-preco, lookback=3 ===


... (12 outputs left out)

  fold 3: 655 tr seqs | 45s


  fold 4: 861 tr seqs | 57s


  fold 5: 1067 tr seqs | 71s



=== Sweep de lookback -- LSTM so-preco ===
             MAE    RMSE      DA     MCC  MCC_sd
lookback                                        
1         0.0122  0.0166  0.4992 -0.0138  0.0682
3         0.0123  0.0168  0.5019 -0.0058  0.0424
5         0.0125  0.0170  0.4957 -0.0034  0.0427
10        0.0125  0.0171  0.5056  0.0054  0.0450
21        0.0127  0.0172  0.4938 -0.0086  0.0567

Salvo: sweep_lookback_preco.parquet


In [4]:
#@title Ablacao -- janelas de lookback {1,3,5,10} -- LSTM + BERTimbau k=8
feat_texto = pd.read_parquet(DIRS['processed'] / 'features_texto_bertimbau.parquet')
feat_texto = feat_texto.reindex(feat_preco.index)
EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']
X_sig = feat_texto[SIG_COLS].values.astype(np.float32)
X_emb = feat_texto[EMB_COLS].values.astype(np.float32)
X_day_txt = np.concatenate([X_price, X_sig], axis=1)

sweep_rows_txt = []
for lb in [1, 3, 5, 10]:
    print(f"=== LSTM + BERTimbau k=8, lookback={lb} ===")
    df = roda_lstm(X_day_txt, y_preco, feat_preco.index, lb, f'lstm_texto_k8_lb{lb}',
                    k_pca=8, X_emb_full=X_emb)
    r, d = resumo_desvio(df)
    sweep_rows_txt.append({'lookback': lb, 'fonte': 'BERTimbau-k8', **r.to_dict(),
                            **{f'{m}_sd': d[m] for m in metric_cols}})

n5 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
r21t, d21t = resumo_desvio(n5)
sweep_rows_txt.append({'lookback': 21, 'fonte': 'BERTimbau-k8', **r21t.to_dict(),
                        **{f'{m}_sd': d21t[m] for m in metric_cols}})

sweep_texto_df = pd.DataFrame(sweep_rows_txt).sort_values('lookback').set_index('lookback')
print("\n=== Sweep de lookback -- LSTM + BERTimbau k=8 ===")
print(sweep_texto_df[['MAE','RMSE','DA','MCC','MCC_sd']].round(4).to_string())
sweep_texto_df.to_parquet(DIRS['reports'] / 'sweep_lookback_texto.parquet')
print("\nSalvo: sweep_lookback_texto.parquet")


=== LSTM + BERTimbau k=8, lookback=1 ===


  fold 1: 252 tr seqs | 17s


  fold 2: 458 tr seqs | 29s


  fold 3: 664 tr seqs | 42s


  fold 4: 870 tr seqs | 54s


  fold 5: 1076 tr seqs | 66s
=== LSTM + BERTimbau k=8, lookback=3 ===


... (13 outputs left out)

  fold 3: 655 tr seqs | 45s


  fold 4: 861 tr seqs | 57s


  fold 5: 1067 tr seqs | 71s



=== Sweep de lookback -- LSTM + BERTimbau k=8 ===
             MAE    RMSE      DA     MCC  MCC_sd
lookback                                        
1         0.0126  0.0170  0.4883  0.0007  0.0601
3         0.0128  0.0174  0.4950 -0.0206  0.0526
5         0.0128  0.0174  0.4957 -0.0116  0.0515
10        0.0130  0.0177  0.4928 -0.0056  0.0553
21        0.0132  0.0181  0.4944 -0.0180  0.0538

Salvo: sweep_lookback_texto.parquet


In [5]:
#@title Resumo consolidado das ablacoes (D vs D-1, sweep de lookback)
print("=== D vs. D-1 (BERTimbau k=8, lookback=21) -- MCC ===")
print(f"  so-preco:  {res_n4['MCC']:+.4f}")
print(f"  texto D:   {res_n5['MCC']:+.4f} +/- {dev_n5['MCC']:.4f}")
print(f"  texto D-1: {r_lag1['MCC']:+.4f} +/- {d_lag1['MCC']:.4f}")

print("\n=== Sweep de lookback -- MCC por janela ===")
comp = sweep_preco_df[['MCC','MCC_sd']].rename(columns={'MCC':'MCC_preco','MCC_sd':'MCC_preco_sd'}).join(
       sweep_texto_df[['MCC','MCC_sd']].rename(columns={'MCC':'MCC_texto','MCC_sd':'MCC_texto_sd'}))
print(comp.round(4).to_string())

comp.to_parquet(DIRS['reports'] / 'sweep_lookback_consolidado.parquet')
print("\nSalvo: sweep_lookback_consolidado.parquet")
print("\nConclusao preliminar (conferir apos rodar): se o MCC permanecer",
      "dentro do desvio entre sementes em TODAS as janelas, o resultado nulo",
      "e robusto a escolha do lookback -- nao e um artefato da janela de 21 dias.")


=== D vs. D-1 (BERTimbau k=8, lookback=21) -- MCC ===
  so-preco:  -0.0086
  texto D:   -0.0180 +/- 0.0538
  texto D-1: -0.0158 +/- 0.0537

=== Sweep de lookback -- MCC por janela ===
          MCC_preco  MCC_preco_sd  MCC_texto  MCC_texto_sd
lookback                                                  
1           -0.0138        0.0682     0.0007        0.0601
3           -0.0058        0.0424    -0.0206        0.0526
5           -0.0034        0.0427    -0.0116        0.0515
10           0.0054        0.0450    -0.0056        0.0553
21          -0.0086        0.0567    -0.0180        0.0538

Salvo: sweep_lookback_consolidado.parquet

Conclusao preliminar (conferir apos rodar): se o MCC permanecer dentro do desvio entre sementes em TODAS as janelas, o resultado nulo e robusto a escolha do lookback -- nao e um artefato da janela de 21 dias.


In [6]:
#@title Ablacao -- anisotropia dos embeddings (whitening) + teste de permutacao
# Motivacao: embeddings de BERT sao conhecidamente anisotropicos (Ethayarajh,
# 2019) -- pares de textos SEM relacao semantica tendem a ter similaridade de
# cosseno bem acima de zero, porque boa parte da "energia" do espaco vive numa
# direcao/subespaco generico compartilhado, nao-informativo. O pipeline
# original (niveis 5/5b, Via 1/1b) aplica PCA direto sobre os embeddings
# CRUS, sem correcao dessa anisotropia -- risco real de que os primeiros
# componentes principais capturem majoritariamente esse artefato, nao sinal
# semantico, empurrando sinal genuino para componentes descartados no corte
# de k=8/k=30.
#
# Verificacao empirica (corpus completo, fora do pipeline por fold):
#   - similaridade de cosseno media entre pares aleatorios de manchetes: 0.66
#     (num espaco isotropico seria ~0). Confirma anisotropia real.
#   - o proprio teste de sanidade do notebook 03 ja mostrava isso sem que
#     tivesse sido notado: sim(bolsa em alta, receita de bolo) = 0.505 --
#     "deve ser BAIXA" no comentario original, mas 0.505 nao e baixo em termos
#     absolutos, so e menor que 0.899 (o par realmente relacionado).
#
# Correcao testada: whitening (decorrelacionar + igualar variancia em todas
# as direcoes, ao molde de BERT-flow) antes de agregar por dia e aplicar PCA.
# Isso e feito aqui de forma EXPLORATORIA (PCA/whitening ajustados uma vez
# sobre a serie completa, nao por fold) -- adequado para diagnostico de
# "existe sinal recuperavel aqui" mas NAO usado para treinar nenhum modelo
# reportado no trabalho (evitaria vazamento se fosse).

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from collections import defaultdict

corpus = pd.read_parquet(DIRS['processed'] / 'corpus_bucketed.parquet').reset_index(drop=True)
feat_preco_diag = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
y_diag = feat_preco_diag['y'].values
ret_hoje_diag = feat_preco_diag['ret_lag0'].values

def whiten(Emb):
    mu = Emb.mean(axis=0)
    cov = np.cov((Emb - mu).T)
    autoval, autovec = np.linalg.eigh(cov)
    autoval = np.clip(autoval, 1e-6, None)
    W = autovec @ np.diag(autoval**-0.5) @ autovec.T
    return (Emb - mu) @ W

def agrega_por_dia(Emb, dias_col):
    idx_por_dia = defaultdict(list)
    for i, d in enumerate(dias_col):
        idx_por_dia[d].append(i)
    dias_unicos = np.array(sorted(pd.unique(dias_col)))
    centroides = np.vstack([Emb[idx_por_dia[d]].mean(axis=0) for d in dias_unicos])
    return pd.DataFrame(centroides, index=pd.DatetimeIndex(dias_unicos))

def diagnostico(feat_dia, nome, k=100):
    X = feat_dia.reindex(feat_preco_diag.index).values.astype(np.float64)
    pca = PCA(n_components=k, random_state=0).fit(X)
    scores = pca.transform(X)
    corrs_pred = [abs(np.corrcoef(scores[:, i], y_diag)[0, 1]) for i in range(k)]
    corrs_cont = [abs(np.corrcoef(scores[:, i], ret_hoje_diag)[0, 1]) for i in range(k)]
    r_pred, r_cont = max(corrs_pred), max(corrs_cont)
    print(f"{nome} (var {pca.explained_variance_ratio_.sum():.1%}, k={k}): "
          f"max|r| preditivo={r_pred:.4f} | max|r| contemporaneo={r_cont:.4f}")
    return r_pred, scores

# --- Similaridade de cosseno media (checagem de anisotropia) ---
E_bert = np.load(DIRS['processed'] / 'emb_bertimbau_raw.npy')
rng = np.random.default_rng(0)
idx_amostra = rng.choice(len(E_bert), size=2000, replace=False)
amostra = E_bert[idx_amostra]
amostra_norm = amostra / np.linalg.norm(amostra, axis=1, keepdims=True)
sims = amostra_norm @ amostra_norm.T
iu = np.triu_indices(len(amostra), k=1)
print(f"Similaridade de cosseno media entre pares aleatorios de manchetes: "
      f"{sims[iu].mean():.3f} (0=isotropico; confirma anisotropia se >> 0)\n")

# --- BERTimbau: cru vs whitened, varios k ---
print("=== BERTimbau ===")
feat_cru = agrega_por_dia(E_bert, corpus['trading_day'].values)
feat_white_bert = agrega_por_dia(whiten(E_bert), corpus['trading_day'].values)
for k in (8, 30, 100, 300):
    diagnostico(feat_cru, f"  cru        k={k}", k=k)
    diagnostico(feat_white_bert, f"  whitened   k={k}", k=k)
print()

# --- FinBERT: cru vs whitened ---
print("=== FinBERT ===")
E_fb = np.load(DIRS['processed'] / 'finbert_emb_raw.npy')
feat_cru_fb = agrega_por_dia(E_fb, corpus['trading_day'].values)
feat_white_fb = agrega_por_dia(whiten(E_fb), corpus['trading_day'].values)
for k in (8, 30, 100, 300):
    diagnostico(feat_cru_fb, f"  cru        k={k}", k=k)
    diagnostico(feat_white_fb, f"  whitened   k={k}", k=k)

print("\nSanidade cruzada: a correlacao CONTEMPORANEA do FinBERT whitened "
      "(k>=30) deve aproximar-se de +0.201 -- a correlacao ja reportada via "
      "a cabeca de sentimento dedicada do FinBERT (Secao 5.4) -- confirmando "
      "que o whitening recupera sinal real, nao ruido.")


Similaridade de cosseno media entre pares aleatorios de manchetes: 0.659 (0=isotropico; confirma anisotropia se >> 0)

=== BERTimbau ===


  cru        k=8 (var 52.3%, k=8): max|r| preditivo=0.0409 | max|r| contemporaneo=0.0315
  whitened   k=8 (var 7.2%, k=8): max|r| preditivo=0.0346 | max|r| contemporaneo=0.0676


  cru        k=30 (var 72.1%, k=30): max|r| preditivo=0.0512 | max|r| contemporaneo=0.0637
  whitened   k=30 (var 16.4%, k=30): max|r| preditivo=0.0640 | max|r| contemporaneo=0.0672


  cru        k=100 (var 88.1%, k=100): max|r| preditivo=0.0871 | max|r| contemporaneo=0.0638
  whitened   k=100 (var 37.2%, k=100): max|r| preditivo=0.0745 | max|r| contemporaneo=0.0768


  cru        k=300 (var 97.8%, k=300): max|r| preditivo=0.0847 | max|r| contemporaneo=0.0881


  whitened   k=300 (var 73.6%, k=300): max|r| preditivo=0.0904 | max|r| contemporaneo=0.0680

=== FinBERT ===


  cru        k=8 (var 86.6%, k=8): max|r| preditivo=0.0761 | max|r| contemporaneo=0.2146
  whitened   k=8 (var 6.9%, k=8): max|r| preditivo=0.0262 | max|r| contemporaneo=0.1869
  cru        k=30 (var 92.4%, k=30): max|r| preditivo=0.0761 | max|r| contemporaneo=0.2146


  whitened   k=30 (var 16.1%, k=30): max|r| preditivo=0.0610 | max|r| contemporaneo=0.1888
  cru        k=100 (var 96.9%, k=100): max|r| preditivo=0.0826 | max|r| contemporaneo=0.2146


  whitened   k=100 (var 36.9%, k=100): max|r| preditivo=0.0600 | max|r| contemporaneo=0.1879


  cru        k=300 (var 99.4%, k=300): max|r| preditivo=0.0823 | max|r| contemporaneo=0.2146


  whitened   k=300 (var 73.3%, k=300): max|r| preditivo=0.0788 | max|r| contemporaneo=0.1886

Sanidade cruzada: a correlacao CONTEMPORANEA do FinBERT whitened (k>=30) deve aproximar-se de +0.201 -- a correlacao ja reportada via a cabeca de sentimento dedicada do FinBERT (Secao 5.4) -- confirmando que o whitening recupera sinal real, nao ruido.


In [7]:
#@title Teste de permutacao -- a correlacao preditiva whitened e real ou artefato de multiplas comparacoes?
# Com k grande (ex.: 300), o MAXIMO de varias correlacoes tende a crescer so
# por efeito de estatistica de ordem, mesmo sob H0 (nenhuma correlacao real).
# Testa isso diretamente: embaralha o alvo (quebra qualquer relacao real,
# preserva a estrutura interna dos componentes) e recalcula o max|r| --
# repete 1000x para construir a distribuicao nula empirica.

K_PERM = 300
pca_perm = PCA(n_components=K_PERM, random_state=0).fit(
    feat_white_bert.reindex(feat_preco_diag.index).values.astype(np.float64))
scores_perm = pca_perm.transform(
    feat_white_bert.reindex(feat_preco_diag.index).values.astype(np.float64))

obs_maxr = max(abs(np.corrcoef(scores_perm[:, i], y_diag)[0, 1]) for i in range(K_PERM))
print(f"max|r| observado (BERTimbau whitened, k={K_PERM}): {obs_maxr:.4f}")

rng = np.random.default_rng(0)
nulos = []
for _ in range(500):
    y_perm = rng.permutation(y_diag)
    nulos.append(max(abs(np.corrcoef(scores_perm[:, i], y_perm)[0, 1]) for i in range(K_PERM)))
nulos = np.array(nulos)
p_valor = (nulos >= obs_maxr).mean()
print(f"nulo empirico (500 permutacoes) -- media: {nulos.mean():.4f} | "
      f"p95: {np.percentile(nulos, 95):.4f}")
print(f"p-valor empirico: {p_valor:.3f}  "
      f"({'NAO significativo -- consistente com ruido' if p_valor > 0.05 else 'significativo'})")

resultado_anisotropia = pd.DataFrame([{
    'fonte': 'BERTimbau', 'k': K_PERM, 'max_r_observado': obs_maxr,
    'p_valor_permutacao': p_valor, 'nulo_media': nulos.mean(), 'nulo_p95': np.percentile(nulos, 95),
}])
resultado_anisotropia.to_parquet(DIRS['reports'] / 'ablacao_anisotropia_whitening.parquet')
print("\nSalvo: ablacao_anisotropia_whitening.parquet")


max|r| observado (BERTimbau whitened, k=300): 0.0904


nulo empirico (500 permutacoes) -- media: 0.0815 | p95: 0.1009
p-valor empirico: 0.178  (NAO significativo -- consistente com ruido)

Salvo: ablacao_anisotropia_whitening.parquet


In [8]:
#@title Ablacao -- qualidade do texto: manchetes-template e linguagem prospectiva/retrospectiva
# Duas hipoteses concretas sobre se a QUALIDADE do texto (nao a compressao)
# estaria suprimindo sinal:
# (1) Colunas fixas recorrentes (ex.: "Os 5 assuntos que vao movimentar o
#     mercado nesta terca-feira") tem embedding quase identico toda semana,
#     independente do conteudo real do dia -- vetores de informacao zero
#     diluindo a media diaria. Testa remover essas manchetes-template.
# (2) A maioria das manchetes e RETROSPECTIVA (recapitula o que ja aconteceu
#     -- reforca o sinal contemporaneo ja conhecido) e poucas sao
#     PROSPECTIVAS (linguagem de previsao/expectativa). Testa se isolar so
#     as prospectivas revela sinal preditivo escondido na maioria retrospectiva.

padroes_template = (r'os \d+ assuntos que v.o (movimentar|agitar) o(s)? mercado(s)? nesta|'
                     r'day trade hoje: o que esperar dos minicontratos')
mask_template = corpus['headline'].astype(str).str.contains(
    padroes_template, case=False, regex=True, na=False)
print(f"Manchetes-template identificadas: {mask_template.sum()} "
      f"({mask_template.mean():.1%} do corpus)\n")

prospectiva = (r'\b(deve|deverá|deveria|vai|irá|espera(-se)?|esperado|previsão|'
               r'previsões|projeta|projeção|estima|amanhã|próximo|próximos|'
               r'tende a|perspectiva|aponta para)\b')
retrospectiva = r'\b(fechou|encerrou|subiu|caiu|recuou|avançou|registrou|teve|ontem)\b'
mask_prosp = corpus['headline'].astype(str).str.contains(prospectiva, case=False, regex=True, na=False)
mask_retro = corpus['headline'].astype(str).str.contains(retrospectiva, case=False, regex=True, na=False)
print(f"Manchetes prospectivas: {mask_prosp.sum()} ({mask_prosp.mean():.1%})")
print(f"Manchetes retrospectivas: {mask_retro.sum()} ({mask_retro.mean():.1%})\n")

def agrega_filtrado_e_testa(mask_manter, nome, k=30, n_perm=300):
    sub_corpus = corpus[mask_manter].reset_index(drop=True)
    sub_E = E_bert[mask_manter.values]
    dias = sub_corpus['trading_day'].values
    idx_por_dia = defaultdict(list)
    for i, d in enumerate(dias):
        idx_por_dia[d].append(i)
    dias_validos = np.array(sorted(idx_por_dia.keys()))
    Ew = whiten(sub_E)
    centroides = np.vstack([Ew[idx_por_dia[d]].mean(axis=0) for d in dias_validos])
    feat = pd.DataFrame(centroides, index=pd.DatetimeIndex(dias_validos))
    feat_alvo = feat_preco_diag.reindex(feat.index)
    X = feat.values.astype(np.float64)
    y_sub = feat_alvo['y'].values
    ok = ~np.isnan(y_sub)
    X, y_sub = X[ok], y_sub[ok]
    kk = min(k, X.shape[0] - 1, X.shape[1])
    pca = PCA(n_components=kk, random_state=0).fit(X)
    scores = pca.transform(X)
    obs = max(abs(np.corrcoef(scores[:, i], y_sub)[0, 1]) for i in range(kk))

    rng_local = np.random.default_rng(0)
    nulos = np.array([
        max(abs(np.corrcoef(scores[:, i], rng_local.permutation(y_sub))[0, 1]) for i in range(kk))
        for _ in range(n_perm)
    ])
    p = (nulos >= obs).mean()
    print(f"{nome}: n_dias={len(y_sub)} | k={kk} | max|r| preditivo={obs:.4f} | "
          f"p-valor (permutacao, {n_perm}x)={p:.3f}")
    return obs, p

print("=== Filtragem de manchetes-template ===")
agrega_filtrado_e_testa(pd.Series(True, index=corpus.index), "COM templates (baseline)")
agrega_filtrado_e_testa(~mask_template, "SEM templates (filtrado)")

print("\n=== Prospectiva vs. retrospectiva ===")
agrega_filtrado_e_testa(mask_prosp, "SO prospectivas")
agrega_filtrado_e_testa(mask_retro, "SO retrospectivas")


Manchetes-template identificadas: 871 (1.5% do corpus)

Manchetes prospectivas: 5506 (9.6%)
Manchetes retrospectivas: 251 (0.4%)

=== Filtragem de manchetes-template ===


C:\Users\GAMER\AppData\Local\Temp\ipykernel_23092\624195851.py:15: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_template = corpus['headline'].astype(str).str.contains(
C:\Users\GAMER\AppData\Local\Temp\ipykernel_23092\624195851.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_prosp = corpus['headline'].astype(str).str.contains(prospectiva, case=False, regex=True, na=False)
C:\Users\GAMER\AppData\Local\Temp\ipykernel_23092\624195851.py:25: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_retro = corpus['headline'].astype(str).str.contains(retrospectiva, case=False, regex=True, na=False)


COM templates (baseline): n_dias=1492 | k=30 | max|r| preditivo=0.0640 | p-valor (permutacao, 300x)=0.347


SEM templates (filtrado): n_dias=1492 | k=30 | max|r| preditivo=0.0619 | p-valor (permutacao, 300x)=0.430

=== Prospectiva vs. retrospectiva ===


SO prospectivas: n_dias=1387 | k=30 | max|r| preditivo=0.0496 | p-valor (permutacao, 300x)=0.867


SO retrospectivas: n_dias=215 | k=30 | max|r| preditivo=0.1732 | p-valor (permutacao, 300x)=0.290


(np.float64(0.1731795845621845), np.float64(0.29))